In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 8))

/tmp/ipykernel_73433/3355534443.py:2: FutureWarning: Use `scanpy.set_figure_params` instead
  sc.settings.set_figure_params(dpi=150, facecolor='white', figsize=(8, 8))


In [3]:
data_dir = os.path.expanduser("../data/151673")
output_dir = os.path.expanduser("../results/figures")
processed_dir = os.path.expanduser("../data/processed")


adata = sc.read_visium(
    path=data_dir,
    count_file="filtered_feature_bc_matrix.h5",
    library_id="151673"
)
adata.var_names_make_unique()

reading ../data/151673/filtered_feature_bc_matrix.h5


/tmp/ipykernel_73433/3052723387.py:6: FutureWarning: Use `squidpy.read.visium` instead.
  adata = sc.read_visium(


 (0:00:00)


/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
print(f"Spatial keys: {list(adata.uns['spatial'].keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")
print(f"Shape: {adata.shape}")

Spatial keys: ['151673']
obsm keys: ['spatial']
Shape: (3639, 33538)


In [5]:
# load ground truth 
meta = pd.read_csv('../data/metadata.tsv', sep='\t', index_col=0)
adata.obs['ground_truth'] = meta.loc[adata.obs_names, 'layer_guess_reordered']

In [6]:
# print coverage 
print(f'{adata.obs['ground_truth'].notna().sum()}/{adata.n_obs}')

3611/3639


In [7]:
adata = adata[adata.obs["ground_truth"].notna()].copy()
adata.obs["ground_truth"] = adata.obs["ground_truth"].astype("category")

In [8]:
# of spot per layer
adata.obs["ground_truth"].value_counts()

ground_truth
Layer3    989
Layer6    692
Layer5    673
WM        513
Layer1    273
Layer2    253
Layer4    218
Name: count, dtype: int64

In [9]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, percent_top=None)

In [10]:
print(f"Total counts: median={adata.obs['total_counts'].median():.0f}, "
      f"mean={adata.obs['total_counts'].mean():.0f}")
print(f"Genes detected: median={adata.obs['n_genes_by_counts'].median():.0f}")
print(f"Mito %: median={adata.obs['pct_counts_mt'].median():.1f}%")

Total counts: median=4138, mean=4601
Genes detected: median=2117
Mito %: median=16.7%


In [11]:
fig, axes = plt.subplots(2, 2, figsize=(16, 16))

sc.pl.spatial(adata, color="total_counts", ax=axes[0, 0], show=False,
              title="Total UMI Counts", spot_size=1.5)
sc.pl.spatial(adata, color="n_genes_by_counts", ax=axes[0, 1], show=False,
              title="Genes Detected", spot_size=1.5)
sc.pl.spatial(adata, color="pct_counts_mt", ax=axes[1, 0], show=False,
              title="Mitochondrial %", spot_size=1.5)
sc.pl.spatial(adata, color="ground_truth", ax=axes[1, 1], show=False,
              title="Ground Truth Layers", spot_size=1.5)

plt.tight_layout()
plt.savefig(os.path.join(output_dir, "spatial_qc_panel.png"), dpi=200, bbox_inches='tight')
plt.close()

/tmp/ipykernel_73433/771718615.py:3: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="total_counts", ax=axes[0, 0], show=False,
/tmp/ipykernel_73433/771718615.py:5: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="n_genes_by_counts", ax=axes[0, 1], show=False,
/tmp/ipykernel_73433/771718615.py:7: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="pct_counts_mt", ax=axes[1, 0], show=False,
/tmp/ipykernel_73433/771718615.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color="ground_truth", ax=axes[1, 1], show=False,


In [12]:
print(f"\nbefore: {adata.n_obs} spots, {adata.n_vars} genes")


before: 3611 spots, 33538 genes


In [13]:
# filtering by spot
sc.pp.filter_cells(adata, min_counts=500)
sc.pp.filter_cells(adata, min_genes=200)
adata = adata[adata.obs["pct_counts_mt"] < 30].copy()

# filtering by gene
sc.pp.filter_genes(adata, min_cells=10)

filtered out 7 cells that have less than 500 counts
filtered out 16971 genes that are detected in less than 10 cells


In [14]:
print(f"after: {adata.n_obs} spots, {adata.n_vars} genes")

after: 3601 spots, 16567 genes


In [16]:
adata.write(os.path.join(processed_dir, "01_qc.h5ad"))